# Airbnb Geospatial Analysis — Data Cleaning

## Objective
This notebook loads and cleans the Airbnb Albany dataset for exploratory analysis and geospatial hotspot detection.

## Tasks
- Load datasets
- Inspect schema and missing values
- Clean pricing columns
- Convert dates
- Handle duplicates and invalid coordinates
- Save processed datasets

In [1]:
import numpy as np 
import pandas as pd 

pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',100)

In [2]:
listings = pd.read_csv("../data/raw/listings.csv.gz")
calendar = pd.read_csv("../data/raw/calendar.csv.gz")
reviews = pd.read_csv("../data/raw/reviews.csv.gz")

In [3]:
print("Listings Shape:", listings.shape)
print("Calendar Shape:", calendar.shape)
print("Reviews Shape:", reviews.shape)

Listings Shape: (453, 85)
Calendar Shape: (165345, 7)
Reviews Shape: (27892, 6)


In [4]:
listings.columns.tolist()

['id',
 'listing_url',
 'scrape_id',
 'last_scraped',
 'source',
 'name',
 'description',
 'neighborhood_overview',
 'picture_url',
 'host_id',
 'host_url',
 'host_profile_id',
 'host_profile_url',
 'host_name',
 'host_since',
 'hosts_time_as_user_years',
 'hosts_time_as_user_months',
 'hosts_time_as_host_years',
 'hosts_time_as_host_months',
 'host_location',
 'host_about',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'host_thumbnail_url',
 'host_picture_url',
 'host_neighbourhood',
 'host_listings_count',
 'host_total_listings_count',
 'host_verifications',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood',
 'neighbourhood_cleansed',
 'neighbourhood_group_cleansed',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bathrooms',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'minimum_nights',
 'maximum_nights',
 'minimum_minimum_nights',
 'maximum_minimum_nights',
 'minimu

In [5]:
calendar.columns.tolist()

['listing_id',
 'date',
 'available',
 'price',
 'adjusted_price',
 'minimum_nights',
 'maximum_nights']

In [6]:
reviews.columns.tolist()

['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments']

In [7]:
listings.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2992450,https://www.airbnb.com/rooms/2992450,20260215064701,2026-02-15,city scrape,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,NaN,https://a0.muscache.com/pictures/44627226/0e72...,4621559,https://www.airbnb.com/users/show/4621559,1462661762635155041,https://www.airbnb.com/users/profile/146266176...,Kenneth,NaN,13,1,13,0,"New York, NY",I am a real down to earth & cool person.,NaN,NaN,NaN,f,NaN,https://a0.muscache.com/im/users/4621559/profi...,NaN,1,NaN,NaN,t,t,NaN,THIRD WARD,NaN,42.65789,-73.75370,Entire rental unit,Entire home/apt,4,1.0,1 bath,2.0,2.0,"[""Smoke alarm"", ""Kitchen"", ""Air conditioning"",...",NaN,28,1125,28,28,1125,1125,28.0,1125.0,NaN,t,0,16,46,321,2026-02-15,9,0,0,276,0,0,NaN,2014-07-01,2022-08-17,3.56,3.44,3.56,4.22,4.56,3.22,3.67,NaN,NaN,1,1,0,0,0.06
1,3820211,https://www.airbnb.com/rooms/3820211,20260215064701,2026-02-15,city scrape,Restored Precinct in Center Sq. w/Parking,Step into the charming and comfy 1BR/1BA apart...,NaN,https://a0.muscache.com/pictures/prohost-api/H...,19648678,https://www.airbnb.com/users/show/19648678,1463087024016718788,https://www.airbnb.com/users/profile/146308702...,Ming,NaN,11,6,11,6,"Albany, NY","Hello! I’m a proud resident of Albany, NY, whe...",NaN,NaN,NaN,t,NaN,https://a0.muscache.com/im/pictures/user/User/...,NaN,7,NaN,NaN,t,t,NaN,SIXTH WARD,NaN,42.65222,-73.76724,Entire rental unit,Entire home/apt,3,1.0,1 bath,1.0,1.0,"[""Smoke alarm"", ""Kitchen"", ""Dishwasher"", ""Wifi...",NaN,1,1125,1,3,1125,1125,2.7,1125.0,NaN,t,0,0,26,301,2026-02-15,317,10,2,256,9,60,NaN,2014-08-15,2026-01-25,4.75,4.88,4.87,4.85,4.81,4.82,4.78,NaN,NaN,7,7,0,0,2.26
2,5651579,https://www.airbnb.com/rooms/5651579,20260215064701,2026-02-15,city scrape,Large studio apt by Capital Center & ESP@,"Spacious studio with hardwood floors, fully eq...",NaN,https://a0.muscache.com/pictures/b3fc42f3-6e5e...,29288920,https://www.airbnb.com/users/show/29288920,1465431505328745177,https://www.airbnb.com/users/profile/146543150...,Gregg,NaN,10,11,10,11,"Albany, NY",I am an Albany native .I have lived in Ireland...,NaN,NaN,NaN,f,NaN,https://a0.muscache.com/im/users/29288920/prof...,NaN,2,NaN,NaN,t,t,NaN,SECOND WARD,NaN,42.64615,-73.75966,Entire rental unit,Entire home/apt,2,1.0,1 bath,0.0,2.0,"[""Smoke alarm"", ""Kitchen"", ""Wifi"", ""Shower gel...",NaN,1,1125,1,2,1125,1125,1.3,1125.0,NaN,t,18,31,35,35,2026-02-15,389,20,2,35,18,120,NaN,2015-05-08,2026-02-06,4.51,

In [8]:
calendar.head()

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,2992450,2026-02-15,f,NaN,NaN,28,1125
1,2992450,2026-02-16,f,NaN,NaN,28,1125
2,2992450,2026-02-17,f,NaN,NaN,28,1125
3,2992450,2026-02-18,f,NaN,NaN,28,1125
4,2992450,2026-02-19,f,NaN,NaN,28,1125


In [9]:
reviews.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,2992450,15066586,2014-07-01,16827297,Kristen,Large apartment; nice kitchen and bathroom. Ke...
1,2992450,21810844,2014-10-24,22648856,Christopher,"This may be a little late, but just to say Ken..."
2,2992450,27434334,2015-03-04,45406,Altay,The apartment was very clean and convenient to...
3,2992450,28524578,2015-03-25,5485362,John,Kenneth was ready when I got there and arrange...
4,2992450,35913434,2015-06-23,15772025,Jennifer,We were pleased to see how 2nd Street and the ...


In [10]:
# missing value check
missing_values = listings.isnull().sum().sort_values(ascending=False)

missing_values[missing_values > 0]

neighborhood_overview           453
host_since                      453
host_acceptance_rate            453
host_response_rate              453
host_response_time              453
host_total_listings_count       453
host_verifications              453
license                         453
instant_bookable                453
calendar_updated                453
estimated_revenue_l365d         453
price                           453
neighbourhood                   453
host_thumbnail_url              453
host_neighbourhood              453
neighbourhood_group_cleansed    453
host_about                      199
host_location                   120
review_scores_value              58
reviews_per_month                58
review_scores_cleanliness        58
review_scores_location           58
review_scores_checkin            58
first_review                     58
last_review                      58
review_scores_rating             58
review_scores_communication      58
review_scores_accuracy      

In [11]:
missing_values = calendar.isnull().sum().sort_values(ascending=False)

missing_values[missing_values > 0]

price             165345
adjusted_price    165345
dtype: int64

In [12]:
missing_values = reviews.isnull().sum().sort_values(ascending=False)

missing_values[missing_values > 0]

comments    8
dtype: int64

In [13]:
for i in  [listings,calendar,reviews]:
    missing_percentage = (
    i.isnull().mean() * 100
    ).sort_values(ascending=False)

    print(missing_percentage[missing_percentage > 0])
    

neighborhood_overview           100.000000
host_since                      100.000000
host_acceptance_rate            100.000000
host_response_rate              100.000000
host_response_time              100.000000
host_total_listings_count       100.000000
host_verifications              100.000000
license                         100.000000
instant_bookable                100.000000
calendar_updated                100.000000
estimated_revenue_l365d         100.000000
price                           100.000000
neighbourhood                   100.000000
host_thumbnail_url              100.000000
host_neighbourhood              100.000000
neighbourhood_group_cleansed    100.000000
host_about                       43.929360
host_location                    26.490066
review_scores_value              12.803532
reviews_per_month                12.803532
review_scores_cleanliness        12.803532
review_scores_location           12.803532
review_scores_checkin            12.803532
first_revie

In [14]:
print("Duplicate Listings:", listings.duplicated().sum())
print("Duplicate Calendar Rows:", calendar.duplicated().sum())
print("Duplicate Reviews:", reviews.duplicated().sum())

Duplicate Listings: 0
Duplicate Calendar Rows: 0
Duplicate Reviews: 0


In [15]:
listings.describe()

,id,scrape_id,neighborhood_overview,host_id,host_profile_id,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_response_time,host_response_rate,host_acceptance_rate,host_thumbnail_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,neighbourhood,neighbourhood_group_cleansed,latitude,longitude,accommodates,bathrooms,bedrooms,beds,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
count,4.530000e+02,4.530000e+02,0.0,4.530000e+02,4.530000e+02,0.0,453.000000,453.000000,453.000000,453.000000,0.0,0.0,0.0,0.0,0.0,453.000000,0.0,0.0,0.0,0.0,453.000000,453.000000,453.000000,437.000000,450.000000,437.00000,0.0,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,0.0,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,0.0,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,0.0,0.0,453.000000,453.000000,453.000000,453.0,395.000000
mean,8.981917e+17,2.026022e+13,NaN,2.505305e+08,1.468835e+18,NaN,6.741722,5.538631,4.139073,5.746137,NaN,NaN,NaN,NaN,NaN,32.200883,NaN,NaN,NaN,NaN,42.659237,-73.776699,3.439294,1.231121,1.597778,1.82151,NaN,5.825607,631.823400,5.825607,8.439294,621.964680,631.823400,6.810155,625.284327,NaN,18.150110,40.211921,63.326711,258.172185,61.571744,13.865342,0.704194,231.086093,13.973510,79.620309,NaN,4.733291,4.771139,4.749671,4.829342,4.838759,4.633924,4.692835,NaN,NaN,7.662252,5.743929,1.918322,0.0,1.840684
std,5.701275e+17,3.910569e-03,NaN,2.013526e+08,1.395864e+16,NaN,3.416767,3.291372,3.007453,3.409320,NaN,NaN,NaN,NaN,NaN,158.916079,NaN,NaN,NaN,NaN,0.010570,0.018555,2.436622,0.639231,1.171485,1.32381,NaN,14.542524,428.346775,14.542524,29.043893,429.733446,428.346775,18.010319,428.157406,NaN,10.458704,20.146678,28.941031,115.813576,114.667091,19.267722,1.363528,100.616393,19.806457,84.195552,NaN,0.382619,0.378788,0.368865,0.329483,0.364384,0.451638,0.403480,NaN,NaN,7.994650,7.567732,3.860962,0.0,1.888654
min,2.992450e+06,2.026022e+13,NaN,6.490680e+05,1.462518e+18,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,42.630660,-73.876490,1.000000,0.000000,0.000000,0.00000,NaN,1.000000,3.000000,1.000000,1.000000,1.000000,3.000000,1.000000,3.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,2.500000,2.330000,2.330000,2.500000,1.000000,2.000000,2.500000,NaN,NaN,1.000000,0.000000,0.000000,0.0,0.030000
25%,5.932721e+17,2.026022e+13,NaN,4.762598e+07,1.463514e+18,NaN,4.000000,3.000000,2.000000,3.000000,NaN,NaN,NaN,NaN,NaN,2.000000,NaN,NaN,NaN,NaN,42.652620,-73.787714,2.000000,1.000000,1.000000,1.00000,NaN,1.000000,365.000000,1.000000,1.000000,365.000000,365.000000,1.000000,365.000000,NaN,11.000000,26.000000,46.000000,175.000000,3.000000,1.000000,0.000000,167.000000,1.000000,6.000000,NaN,4.670000,4.750000,4.685000,4.810000,4.840000,4.500000,4.630000,NaN,NaN,1.000000,1.000000,0.000000,0.0,0.425000
50%,1.020519e+18,2.026022e+13,NaN,2.329679e+08,1.468114e+18,NaN,7.000000,6.000000,4.000000,6.000000,NaN,NaN,NaN,NaN,NaN,6.000000,NaN,NaN,NaN,NaN,42.657620,-73.774560,2.000000,1.000000,1.000000,1.00000,NaN,1.000000,365.0

In [16]:
listings[['latitude', 'longitude']].isnull().sum()

latitude     0
longitude    0
dtype: int64

### data cleaning

In [17]:
columns_to_drop = [
    'neighborhood_overview',
    'host_since',
    'host_acceptance_rate',
    'host_response_rate',
    'host_response_time',
    'host_total_listings_count',
    'host_verifications',
    'license',
    'instant_bookable',
    'calendar_updated',
    'estimated_revenue_l365d',
    'price',
    'neighbourhood',
    'host_thumbnail_url',
    'host_neighbourhood',
    'neighbourhood_group_cleansed'
]


important_columns = [
    'id',
    'host_id',
    'host_name',
    'host_location',
    'host_is_superhost',

    'neighbourhood_cleansed',

    'latitude',
    'longitude',

    'property_type',
    'room_type',

    'accommodates',
    'bathrooms',
    'bathrooms_text',
    'bedrooms',
    'beds',

    'amenities',

    'minimum_nights',
    'maximum_nights',

    'availability_30',
    'availability_60',
    'availability_90',
    'availability_365',

    'number_of_reviews',
    'number_of_reviews_ltm',
    'number_of_reviews_l30d',

    'estimated_occupancy_l365d',

    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_cleanliness',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_location',
    'review_scores_value',

    'reviews_per_month',

    'first_review',
    'last_review',

    'calculated_host_listings_count'
]


In [18]:
listings.drop(columns=columns_to_drop, inplace=True)

In [19]:
listings = listings.rename(columns={
    'id': 'listing_id',
    'neighbourhood_cleansed': 'neighborhood'
})

In [20]:
date_columns = [
    'first_review',
    'last_review'
]

for col in date_columns:
    listings[col] = pd.to_datetime(
        listings[col],
        format='%Y-%m-%d',
        errors='coerce'
    )

calendar['date'] = pd.to_datetime(
    calendar['date'],
    format='%Y-%m-%d',
    errors='coerce'
)


In [21]:
reviews['date'] = pd.to_datetime(
    reviews['date'],
    format='%Y-%m-%d',
    errors='coerce'
)

In [22]:
calendar.head(10)

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,2992450,2026-02-15,f,NaN,NaN,28,1125
1,2992450,2026-02-16,f,NaN,NaN,28,1125
2,2992450,2026-02-17,f,NaN,NaN,28,1125
3,2992450,2026-02-18,f,NaN,NaN,28,1125
4,2992450,2026-02-19,f,NaN,NaN,28,1125
5,2992450,2026-02-20,f,NaN,NaN,28,1125
6,2992450,2026-02-21,f,NaN,NaN,28,1125
7,2992450,2026-02-22,f,NaN,NaN,28,1125
8,2992450,2026-02-23,f,NaN,NaN,28,1125
9,2992450,2026-02-24,f,NaN,NaN,28,1125


In [23]:
calendar.info()

<class 'pandas.DataFrame'>
RangeIndex: 165345 entries, 0 to 165344
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   listing_id      165345 non-null  int64         
 1   date            165345 non-null  datetime64[us]
 2   available       165345 non-null  str           
 3   price           0 non-null       float64       
 4   adjusted_price  0 non-null       float64       
 5   minimum_nights  165345 non-null  int64         
 6   maximum_nights  165345 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(3), str(1)
memory usage: 8.8 MB


In [24]:
reviews = reviews.dropna(subset=['comments'])

In [25]:
reviews['comments'] = (
    reviews['comments']
    .str.lower()
    .str.strip()
)

In [26]:
bool_columns = [
    'host_is_superhost',
    'host_has_profile_pic',
    'host_identity_verified',
    'has_availability'
]

for col in bool_columns:
    listings[col] = listings[col].map({
        't': True,
        'f': False
    })

In [27]:
import ast

listings['amenities_count'] = listings['amenities'].apply(
    lambda x: len(ast.literal_eval(x))
    if pd.notnull(x) else 0
)

In [28]:
listings['occupancy_rate'] = (
    listings['estimated_occupancy_l365d'] / 365
)

In [29]:
neighborhood_summary = (
    listings
    .groupby('neighborhood')
    .agg({
        'listing_id': 'count',
        'occupancy_rate': 'mean',
        'number_of_reviews': 'mean',
        'review_scores_rating': 'mean'
    })
    .rename(columns={
        'listing_id': 'listing_count'
    })
)

In [30]:
calendar.head()

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,2992450,2026-02-15,f,NaN,NaN,28,1125
1,2992450,2026-02-16,f,NaN,NaN,28,1125
2,2992450,2026-02-17,f,NaN,NaN,28,1125
3,2992450,2026-02-18,f,NaN,NaN,28,1125
4,2992450,2026-02-19,f,NaN,NaN,28,1125


In [37]:
calendar.info()

<class 'pandas.DataFrame'>
RangeIndex: 165345 entries, 0 to 165344
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   listing_id      165345 non-null  int64         
 1   date            165345 non-null  datetime64[us]
 2   available       165345 non-null  str           
 3   price           0 non-null       float64       
 4   adjusted_price  0 non-null       float64       
 5   minimum_nights  165345 non-null  int64         
 6   maximum_nights  165345 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(3), str(1)
memory usage: 8.8 MB


In [31]:
listings.head()

,listing_id,listing_url,scrape_id,last_scraped,source,name,description,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_is_superhost,host_picture_url,host_listings_count,host_has_profile_pic,host_identity_verified,neighborhood,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,amenities_count,occupancy_rate
0,2992450,https://www.airbnb.com/rooms/2992450,20260215064701,2026-02-15,city scrape,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,https://a0.muscache.com/pictures/44627226/0e72...,4621559,https://www.airbnb.com/users/show/4621559,1462661762635155041,https://www.airbnb.com/users/profile/146266176...,Kenneth,13,1,13,0,"New York, NY",I am a real down to earth & cool person.,False,https://a0.muscache.com/im/users/4621559/profi...,1,True,True,THIRD WARD,42.65789,-73.75370,Entire rental unit,Entire home/apt,4,1.0,1 bath,2.0,2.0,"[""Smoke alarm"", ""Kitchen"", ""Air conditioning"",...",28,1125,28,28,1125,1125,28.0,1125.0,True,0,16,46,321,2026-02-15,9,0,0,276,0,0,2014-07-01,2022-08-17,3.56,3.44,3.56,4.22,4.56,3.22,3.67,1,1,0,0,0.06,8,0.000000
1,3820211,https://www.airbnb.com/rooms/3820211,20260215064701,2026-02-15,city scrape,Restored Precinct in Center Sq. w/Parking,Step into the charming and comfy 1BR/1BA apart...,https://a0.muscache.com/pictures/prohost-api/H...,19648678,https://www.airbnb.com/users/show/19648678,1463087024016718788,https://www.airbnb.com/users/profile/146308702...,Ming,11,6,11,6,"Albany, NY","Hello! I’m a proud resident of Albany, NY, whe...",True,https://a0.muscache.com/im/pictures/user/User/...,7,True,True,SIXTH WARD,42.65222,-73.76724,Entire rental unit,Entire home/apt,3,1.0,1 bath,1.0,1.0,"[""Smoke alarm"", ""Kitchen"", ""Dishwasher"", ""Wifi...",1,1125,1,3,1125,1125,2.7,1125.0,True,0,0,26,301,2026-02-15,317,10,2,256,9,60,2014-08-15,2026-01-25,4.75,4.88,4.87,4.85,4.81,4.82,4.78,7,7,0,0,2.26,33,0.164384
2,5651579,https://www.airbnb.com/rooms/5651579,20260215064701,2026-02-15,city scrape,Large studio apt by Capital Center & ESP@,"Spacious studio with hardwood floors, fully eq...",https://a0.muscache.com/pictures/b3fc42f3-6e5e...,29288920,https://www.airbnb.com/users/show/29288920,1465431505328745177,https://www.airbnb.com/users/profile/146543150...,Gregg,10,11,10,11,"Albany, NY",I am an Albany native .I have lived in Ireland...,False,https://a0.muscache.com/im/users/29288920/prof...,2,True,True,SECOND WARD,42.64615,-73.75966,Entire rental unit,Entire home/apt,2,1.0,1 bath,0.0,2.0,"[""Smoke alarm"", ""Kitchen"", ""Wifi"", ""Shower gel...",1,1125,1,2,1125,1125,1.3,1125.0,True,18,31,35,35,2026-02-15,389,20,2,35,18,120,2015-05-08,2026-02-06,4.51,4.61,4.45,4.81,4.86,4.76,4.63,2,1,1,0,2.96,40,0.328767
3,6015313,https://www.airbnb.com/rooms/6015313,20260215064701,2026-02-15,city scrape,Comfortable 3BR Home w/ Central Air | Pine Hills,Updated 3-bedroom home in Pine Hills near hosp...,https://a0.muscache.com/pictures/hosting/Hosti...,31223807,https://www.airbnb.com/users/show/31223807,1465459398999270556,https://www.airbnb.com/

In [32]:
reviews.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,2992450,15066586,2014-07-01,16827297,Kristen,large apartment; nice kitchen and bathroom. ke...
1,2992450,21810844,2014-10-24,22648856,Christopher,"this may be a little late, but just to say ken..."
2,2992450,27434334,2015-03-04,45406,Altay,the apartment was very clean and convenient to...
3,2992450,28524578,2015-03-25,5485362,John,kenneth was ready when i got there and arrange...
4,2992450,35913434,2015-06-23,15772025,Jennifer,we were pleased to see how 2nd street and the ...


In [34]:
listings.to_csv(
    "../data/processed/listings_clean.csv",
    index=False
)

In [35]:
calendar.to_csv(
    "../data/processed/calendar_clean.csv",
    index=False
)

In [36]:
reviews.to_csv(
    "../data/processed/reviews_clean.csv",
    index=False
)